<a href="https://colab.research.google.com/github/slab1/mira-training-scripts/blob/main/axolotl_mira_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Axolotl Mira Training - Colab

Train Hoodx/mira-agent-instinct with Axolotl on Colab T4

In [ ]:
!pip install -q axolotl datasets huggingface_hub

In [ ]:
import os
os.makedirs('data', exist_ok=True)
!wget -q https://raw.githubusercontent.com/slab1/mira-training-scripts/main/data/train_axolotl_merged.jsonl -O data/train_axolotl_merged.jsonl

In [ ]:
!wget -q https://raw.githubusercontent.com/slab1/mira-training-scripts/main/mira_axolotl.yml -O mira_axolotl.yml

In [ ]:
import yaml
cfg = yaml.safe_load(open('mira_axolotl.yml'))
cfg['datasets'][0]['path'] = 'data/train_axolotl_merged.jsonl'
with open('mira_axolotl.yml','w') as f:
    yaml.dump(cfg, f)
print('Config updated')

In [ ]:
!axolotl train mira_axolotl.yml

In [3]:
import os

readme_path = '/content/outputs/mira-axolotl/checkpoint-1500/README.md'
if os.path.exists(readme_path):
    with open(readme_path, 'r') as f:
        print(f.read())
else:
    print('README.md not found at the checkpoint path.')

---
base_model: unsloth/Qwen3-8B-unsloth-bnb-4bit
library_name: peft
pipeline_tag: text-generation
tags:
- base_model:adapter:unsloth/Qwen3-8B-unsloth-bnb-4bit
- lora
- sft
- transformers
- trl
- unsloth
---

# Model Card for Model ID

<!-- Provide a quick summary of what the model is/does. -->



## Model Details

### Model Description

<!-- Provide a longer summary of what this model is. -->



- **Developed by:** [More Information Needed]
- **Funded by [optional]:** [More Information Needed]
- **Shared by [optional]:** [More Information Needed]
- **Model type:** [More Information Needed]
- **Language(s) (NLP):** [More Information Needed]
- **License:** [More Information Needed]
- **Finetuned from model [optional]:** [More Information Needed]

### Model Sources [optional]

<!-- Provide the basic links for the model. -->

- **Repository:** [More Information Needed]
- **Paper [optional]:** [More Information Needed]
- **Demo [optional]:** [More Information Needed]

## Uses

<!-- Address

In [4]:
import os

output_dir = 'outputs/mira-axolotl'
if os.path.exists(output_dir):
    print("Contents of output directory:")
    for item in sorted(os.listdir(output_dir)):
        print(f" - {item}")
else:
    print(f"Directory {output_dir} does not exist yet.")

Contents of output directory:
 - README.md
 - checkpoint-1000
 - checkpoint-1500
 - checkpoint-1875
 - checkpoint-387
 - checkpoint-500


In [5]:
from huggingface_hub import notebook_login
notebook_login()

In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import gc

# Clean up GPU and RAM memory before loading
gc.collect()
torch.cuda.empty_cache()

base_model_name = "Qwen/Qwen2.5-7B-Instruct"
adapter_dir = "outputs/mira-axolotl/checkpoint-1875"
repo_id = "Hoodx/mira-agent-instinct-axolotl-merged"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.push_to_hub(repo_id, private=False)

print("Loading base model on GPU...")
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Loading PEFT adapter...")
model = PeftModel.from_pretrained(base_model, adapter_dir)

print("Merging model and adapter...")
merged_model = model.merge_and_unload()

print(f"Pushing merged model to Hugging Face Hub: {repo_id}...")
merged_model.push_to_hub(repo_id, private=False)
print("Upload complete successfully!")

Loading tokenizer...


README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpyv2csr3s/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

...

Loading base model on GPU...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading PEFT adapter...


RuntimeError: Error(s) in loading state_dict for PeftModelForCausalLM:
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight: copying a param with shape torch.Size([16, 4096]) from checkpoint, the shape in current model is torch.Size([16, 3584]).
	size mismatch for base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight: copying a param with shape torch.Size([4096, 16]) from checkpoint, the shape in current model is torch.Size([3584, 16]).
	size mismatch for base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight: copying a param with shape torch.Size([16, 4096]) from checkpoint, the shape in current model is torch.Size([16, 3584]).
	size mismatch for base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight: copying a param with shape torch.Size([1024, 16]) from checkpoint, the shape in current model is torch.Size([512, 16]).
	size mismatch for base_model.model.model.layers.0.self_attn.v_proj.lora_A.default

In [1]:
import torch
from peft import PeftModel, PeftConfig

adapter_dir = "outputs/mira-axolotl/checkpoint-1875"
repo_id = "Hoodx/mira-agent-instinct-axolotl-merged"  # Using the same repo target, or you can adjust it

print(f"Loading adapter configuration and weights from {adapter_dir}...")
# Load the adapter directly
model = PeftModel.from_pretrained(
    model=torch.nn.Module(), # We can push the raw PeftModel weights/config directly to the hub
    model_id=adapter_dir
)

print(f"Pushing LoRA adapter weights and config to Hugging Face Hub: {repo_id}...")
model.push_to_hub(repo_id, private=False)
print("Upload of LoRA adapter completed successfully!")

Loading adapter configuration and weights from outputs/mira-axolotl/checkpoint-1875...


NoMatchingPeftModuleError: No modules were targeted for adaptation. This might be caused by a combination of mismatched target modules and excluded modules. Please check your `target_modules` and `exclude_modules` configuration. You may also have only targeted modules that are marked to be saved (`modules_to_save`).